In [39]:
%matplotlib inline
import matplotlib.pyplot as plt
import pyneb as pn
from astropy.cosmology import FlatLambdaCDM
cosmo = FlatLambdaCDM(H0=70, Om0=0.3)

import numpy as np
import matplotlib.pyplot as plt
import astropy
from astropy.io import fits
from scipy.interpolate import Akima1DInterpolator
from scipy import optimize as opt
import sys
import emcee
import numpy as np
from scipy.optimize import curve_fit
from astropy.table import Table
import astropy.units as u
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
from astropy.io import fits
import time
from astropy.cosmology import FlatLambdaCDM
import astropy.units as u
from scipy import signal
import matplotlib.pyplot as plt
import warnings
import pandas as pd

In [8]:
RUBIES_and_CEERS_table = Table.read('MASTER_RUBIES_AND_CEERS_TABLE.fits')
flux = RUBIES_and_CEERS_table['flux']
source_name = RUBIES_and_CEERS_table['filename']
flux_error = RUBIES_and_CEERS_table['flux_error']
rest_frame = RUBIES_and_CEERS_table['rest_frame_wavelength']
flags = RUBIES_and_CEERS_table['flags']
obs_wavelength = RUBIES_and_CEERS_table['wavelength']
which_catalog = RUBIES_and_CEERS_table['catalog']
wavelength_ang = RUBIES_and_CEERS_table['obs_wavelength_in_ang']
flux_lambda = RUBIES_and_CEERS_table['flux_lambda']
rest_frame_in_ang = RUBIES_and_CEERS_table['rest_frame_wavelength_in_ang']
flux_lambda_cleaned_for_each_spectra = RUBIES_and_CEERS_table['flux_lambda_cleaned']
flux_error_lambda_cleaned = RUBIES_and_CEERS_table['flux_error_lambda_cleaned']
redshift = RUBIES_and_CEERS_table['redshift']
rest_frame_wavelength_cleaned = RUBIES_and_CEERS_table['rest_frame_wavelength_cleaned']
obs_wavelength_cleaned = RUBIES_and_CEERS_table['obs_wavelength_cleaned']
new_redshifts = RUBIES_and_CEERS_table['new_rubies_redshifts'] 

In [9]:
h_beta = 4861.333
h_alpha = 6562.819
h_gamma = 4340.1
OIII4960 = 4958.911
OIII5007 = 5006.843
OIII4363 = 4363.210
SII6716 = 6716.440
SII6731 = 6730.810
OII3726= 3726.032
OII3729= 3728.815

In [15]:
def h_beta_sfr(luminosity_h_beta):
    sfr = 2.86 * (luminosity_h_beta / (1.26 * (10**41)) )
    return sfr

def h_alpha_sfr(luminosity_h_alpha):
    sfr =  luminosity_h_alpha / (1.26 * (10**41)) 
    return sfr
#https://iopscience.iop.org/article/10.1086/305588/pdf

def flux_to_luminosity(flux, redshift):
    distance = cosmo.luminosity_distance(redshift).to(u.cm).value
    lum = 4*np.pi * ((distance)**2 ) * flux
    return lum
    #the 4pi comes from the flux and luminsosity astro equation 

In [11]:
def wavelength_exists(array, wavelength):
    #array = eq.cleaned
    #wavelength = halpha or hbeta
    idx = np.where((array > wavelength - 10) & (array < wavelength + 10))
    if len(flux_cleaned[idx]) == 0:
        return False
    else:
        return True

## EMCEE W/ REDUCED CHI SQUARE FITTING

In [44]:
def gaussian(x, A, mu, sigma):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2))

# Line model including contxainuum
def line(x, b):
    return np.ones(len(x)) * b

def line_model(x, A, mu, sigma, b):
    return gaussian(x, A, mu, sigma) + line(x, b)

def log_likelihood(theta, x, y, yerr):
    model = line_model(x, *theta)
    lnL = -0.5 * np.sum((y - model)**2 / yerr**2)
    return lnL

def log_prior(theta, wave_center, Amp_max):
    A, mu, sigma, b = theta
    left_mu = wave_center - 50
    right_mu = wave_center + 50
    min_A = 0
    max_A = Amp_max * 2
    sigma_window_left = .5
    sigma_window_right = 50
    if (0 < A < max_A) & (left_mu <= mu <= right_mu) & (sigma_window_left <= sigma < sigma_window_right):# & (b > 0):
        return 0.0
    else:
        return -np.inf

def log_probability(theta, x, y, yerr, wave_center, Amp_max):
    lp = log_prior(theta, wave_center, Amp_max)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta, x, y, yerr)

# Function to fit the Hα and Hβ lines
def fitting_line(wave, flux, flux_err, line_center, window_wavelength, diagnose=False):
    min_window = line_center - window_wavelength
    max_window = line_center + window_wavelength
    indx = (wave >= min_window) & (wave <= max_window)

    
    spec_window = flux[indx]
    wave_window = wave[indx]
    err_spec_window = flux_err[indx]

    # Initial guess for the curve fit
    guess_A = np.abs(np.max(spec_window))
    guess_mu = line_center
    guess_sigma = 20
    guess_b = np.abs(np.median(spec_window))
    
    low_bounds = [0, min_window, 0, -guess_b]
    high_bounds = [2 * guess_A, max_window, 200, 2 * guess_b]
    popt, _ = curve_fit(line_model, wave_window, spec_window, p0=[guess_A, guess_mu, guess_sigma, guess_b],
                        bounds=(low_bounds, high_bounds))
    if diagnose == True: 
        plotting_code = line_model(wave_window,popt[0],popt[1],popt[2],popt[3])
#         plt.figure()
#         plt.plot(wave_window,plotting_code,c='cadetblue',label='model')
#         plt.plot(wave_window,spec_window,c='purple',label='data')
#         plt.axvline(min_window)
#         plt.axvline(max_window)

#         plt.legend()
#         plt.show()
    fluxes_emcee = popt[0] * popt[2] * np.sqrt(2 * np.pi)
    
   # return fluxes_emcee
    return popt



def emcee_fit(wave, flux, flux_err, line_center, window_wavelength, 
                 diagnose = False,save=True, filename = 'Emcee_Chains_Galaxy.txt'):
    
    result = fitting_line(wave, flux, flux_err, line_center, window_wavelength, diagnose=True)
    
    #getting the results from the initial fit to then pass into emcee
    guess_A = result[0]
    guess_mu = result[1]
    guess_sigma = result[2]
    guess_b = result[3]
    
    
    #making walkers so that we can use emcee to explore the parameter space
    #centered on the best results from minimization
    amp_jump = np.random.normal(loc = guess_A,            #centered on best A from minimization
                                scale = guess_A/10,       #can wander 1/10 of the value of A
                                size = 32).reshape(-1, 1) 
    
    wavelength_jump = np.random.normal(loc = guess_mu,    #centered on best mu from minimization
                                       scale = 50,      #can wander +/- 0.005 microns 
                                       size = 32).reshape(-1, 1)
    
    sigma_jump = np.random.normal(loc = guess_sigma, scale = 20, size = 32).reshape(-1, 1)

    
    powerb = np.log10(np.abs(guess_b))
    
    b_jump = np.random.normal(loc = guess_b, scale = 1*10**powerb, size = 32).reshape(-1, 1)

    
    #################
    # Diagnostic plotting to see if the parameters were jumping to large values
    # The should be concentrated near their best fit results values
    #################
    if diagnose == True:
        print('Checking the Walker Jumps')
        fig, ax = plt.subplots(nrows = 2, ncols = 2, constrained_layout = True)
        
        ax[0, 0].hist(amp_jump)
        ax[0, 0].set_xlabel('Amplitude')
        
        ax[0, 1].hist(wavelength_jump)
        ax[0, 1].set_xlabel(r'$\mu$')
        
        ax[1, 0].hist(sigma_jump)
        ax[1, 0].set_xlabel(r'$\sigma$')
        
        ax[1, 1].hist(b_jump)
        ax[1, 1].set_xlabel('b')
        
        plt.show()
    

    #stacking along the columns
    starting_walkers = np.hstack((amp_jump,
                                  wavelength_jump, 
                                  sigma_jump, 
                                  #m_jump, 
                                  b_jump))

    #initializing window for emcee around the best result mu
    emcee_window = window_wavelength 
    emcee_indx = np.where((wave >= (line_center - emcee_window)) & 
                          (wave <= (line_center + emcee_window)))[0]

    #emcee subsections
    emcee_spec = flux[emcee_indx]
    emcee_wave = wave[emcee_indx]
    emcee_err = flux_err[emcee_indx]


    
    #plotting initial guess
    if diagnose == True:
        xarr = np.linspace(emcee_wave[0], emcee_wave[-1], 100)
        
        plt.figure()
        plt.title('Input Emcee Spectra and Emcee Fit')
        plt.step(emcee_wave, emcee_spec, color = 'black', alpha = 0.5, label = 'Data', where='mid')
        plt.scatter(emcee_wave, emcee_spec, color = 'black')
        plt.plot(xarr, line_model(xarr, *result), label = 'initial guess', color='plum')
        plt.xlabel(r'Wavelength [$\mu$m]')
        plt.ylabel('Flux')
        plt.legend()
        plt.show()
    
    
    ###########
    #NOTE:
    #need to change output name everytime you run otherwise it will overwrite
    ###########
    
    #saves the input emcee spectra
    emcee_spec_matrix = np.c_[emcee_wave, emcee_spec, emcee_err]
    #np.savetxt(f'Emcee_Spectra.txt', emcee_spec_matrix)

    #initializing walker positions
    pos = starting_walkers
    nwalkers, ndim = pos.shape

    #initializing sampler
    sampler = emcee.EnsembleSampler(nwalkers, ndim, log_probability, 
                                    args=(emcee_wave, emcee_spec, emcee_err, guess_mu, guess_A))

    #running it
    sampler.run_mcmc(pos, 3000, progress=False)

    #getting values back
    #samples = sampler.get_chain()
    flat_samples = sampler.get_chain(flat=True)
    LnL_chain = sampler.flatlnprobability
    burn_in = 2000 
    
    emcee_df = pd.DataFrame()
    emcee_df['A'] = flat_samples[burn_in:, 0]
    emcee_df['mu'] = (flat_samples[burn_in:, 1] ) 
    emcee_df['sigma'] = flat_samples[burn_in:, 2]
    emcee_df['b'] = flat_samples[burn_in:, 3]
    emcee_df['LnL'] = LnL_chain[burn_in:]

    emcee_df = emcee_df[np.isfinite(emcee_df.LnL.values)]

    
    #----REDUCED CHI SQUARED -- TO CHECK IF MODEL FIT IS GOOD
    params = emcee_df.quantile(q=0.5).values[:-1] 
    best_fit_sigma = emcee_df.quantile(q=0.5).values[2] 
    best_fit_mu = emcee_df.quantile(q=0.5).values[1] 
    new_window_plus = best_fit_mu + 5*best_fit_sigma
    new_window_minus = best_fit_mu - 5*best_fit_sigma
    mask = (emcee_wave>new_window_minus) & (emcee_wave<new_window_plus)
    model = line_model(emcee_wave[mask], *params)
    chi_square = np.sum(((model - emcee_spec[mask]) ** 2) / (emcee_err[mask] ** 2))
    n = len(emcee_wave[mask]) 
    degrees_of_freedom = n - 4
    reduced_chi_square = chi_square / degrees_of_freedom
    print(reduced_chi_square)

    plt.figure()
    plt.plot(emcee_wave[mask],emcee_spec[mask])
    plt.errorbar(emcee_wave[mask],emcee_spec[mask],yerr=emcee_err[mask] ,fmt='none' )
    plt.plot(emcee_wave[mask], model,c='plum')
    plt.title('reduced chi squared fit')
    #plt.show()
    pdf.savefig()
    plt.close() 
    #----



    fluxes_emcee = (emcee_df['A']) * (emcee_df['sigma']) * np.sqrt(2 * np.pi)
   
    emcee_df['Fluxes'] = fluxes_emcee 
    
    if diagnose == True:
        
        print('Checking Prameter Posterior Distributions')
        fig, ax = plt.subplots(nrows = 2, ncols = 2, constrained_layout = True)
        
        emcee_df.A.hist(ax = ax[0, 0])
        emcee_df.mu.hist(ax = ax[0, 1])
        emcee_df.sigma.hist(ax = ax[1, 0])
        #emcee_df.m.hist(ax = ax[1, 0])
        emcee_df.b.hist(ax = ax[1, 1])
        
        plt.show()
    
    if diagnose == True:
        xarr = np.linspace(emcee_wave[0], emcee_wave[-1], 100)
        max_spec = np.amax(emcee_spec)
        
        plt.figure()
        plt.title('Input Emcee Spectra and Emcee Fit')
        plt.step(emcee_wave, emcee_spec, color = 'black', alpha = 0.5, label = 'Data',where='mid')
        plt.errorbar(emcee_wave,emcee_spec, yerr = emcee_err,fmt="none")
        plt.scatter(emcee_wave, emcee_spec, color = 'black')
        plt.plot(xarr, line_model(xarr, *emcee_df.quantile(q = 0.5).values[:-2]), label = 'Model')
        
        lower_model = line_model(xarr, *emcee_df.quantile(q = 0.16).values[:-2])
        upper_model = line_model(xarr, *emcee_df.quantile(q = 0.84).values[:-2])   
        plt.fill_between(xarr, lower_model, upper_model, color='gray', alpha=0.5)
        plt.ylim(-2*max_spec,2*max_spec)
        plt.xlabel(r'Wavelength [$\mu$m]')
        plt.ylabel('Flux')
        plt.legend()
       # plt.show()
        pdf.savefig()
        plt.close() 

    
    if diagnose == True:
        plt.figure()
        #xarr = np.linspace(emcee_wave[0], emcee_wave[-1], 100)
        plt.title('Residual (Data - Model)')
        #plt.plot(emcee_wave, emcee_spec, color = 'black', alpha = 0.5, label = 'Data')
        #plt.scatter(emcee_wave, emcee_spec, color = 'black')
        plt.plot(emcee_wave, line_model(emcee_wave, *emcee_df.quantile(q = 0.5).values[:-2])-emcee_spec, label = 'Model')
        plt.xlabel(r'Wavelength [$\mu$m]')
        plt.ylabel('Residual Flux')
        plt.legend()
        print(np.abs( np.mean(line_model(emcee_wave, *emcee_df.quantile(q = 0.5).values[:-2])-emcee_spec)))
        plt.show()
    ###########
    #NOTE:
    #need to also give the filename argument otherwise it will overwrite the default file
    ###########
    if save == True:
        emcee_df.to_csv(filename, sep = ' ')
        
    else:
        return emcee_df['Fluxes']

- make massive PDF file of reduced chi square plot + emcee fit
- in the title of each source, save an ID or some way to classify them
- then we can visually inspect each source and just flag the bad ones (i.e. make an array of Trues this way we can just change to False if need be)
- also make a FINAL flux column of both flux_instrinsic (dust_corrected) [ratio > 1] and flux (not dust corrected) [ratio<1]


## SFR CALCULATION

In [28]:
sfr = []
um_to_ang=10000

for i, r in enumerate(new_redshifts):
    if flags[i] == True:
        file_name = source_name[i]
        after_wavelength = rest_frame_wavelength_cleaned[i] 
        before_flux = flux_lambda_cleaned_for_each_spectra[i] * (1+r)
        before_flux_error = flux_error_lambda_cleaned[i] * (1+r)
        mask = ((np.isfinite(before_flux) )) & ((np.isfinite(before_flux_error) )) 
     
        flux_cleaned = before_flux[mask]
        flux_error_cleaned = before_flux_error[mask]
        equation_cleaned = (after_wavelength[mask])
        window_wavelength = 70   

        if r<6.7:
            if (((np.amax(equation_cleaned)) >= h_alpha) & wavelength_exists(equation_cleaned, h_alpha) ):
                Halpha_fit = emcee_fit(equation_cleaned, flux_cleaned, flux_error_cleaned, h_alpha, window_wavelength,diagnose= False,save=False)
                h_alpha_sfr_value = h_alpha_sfr(flux_to_luminosity(Halpha_fit,r))
                sfr.append(h_alpha_sfr_value)    
             
            else:
                sfr.append(0)
   
        elif r >6.7:
            if (((np.amax(equation_cleaned)) >= h_beta) & wavelength_exists(equation_cleaned, h_beta))  :
                Hbeta_fit = emcee_fit(equation_cleaned, flux_cleaned, flux_error_cleaned, h_beta, window_wavelength,diagnose= False,save=False)
                h_beta_sfr_value = h_beta_sfr(flux_to_luminosity(Hbeta_fit,r))
                sfr.append(h_beta_sfr_value)
             
            else:
                sfr.append(0)

    else:
        sfr.append(0)




/opt/anaconda3/lib/python3.11/site-packages/emcee/moves/red_blue.py:99: RuntimeWarning: invalid value encountered in scalar subtract
  lnpdiff = f + nlp - state.log_prob[j]
/opt/anaconda3/lib/python3.11/site-packages/emcee/moves/red_blue.py:99: RuntimeWarning: invalid value encountered in scalar subtract
  lnpdiff = f + nlp - state.log_prob[j]
/opt/anaconda3/lib/python3.11/site-packages/emcee/moves/red_blue.py:99: RuntimeWarning: invalid value encountered in scalar subtract
  lnpdiff = f + nlp - state.log_prob[j]
/opt/anaconda3/lib/python3.11/site-packages/emcee/moves/red_blue.py:99: RuntimeWarning: invalid value encountered in scalar subtract
  lnpdiff = f + nlp - state.log_prob[j]
/opt/anaconda3/lib/python3.11/site-packages/emcee/moves/red_blue.py:99: RuntimeWarning: invalid value encountered in scalar subtract
  lnpdiff = f + nlp - state.log_prob[j]
/opt/anaconda3/lib/python3.11/site-packages/emcee/moves/red_blue.py:99: RuntimeWarning: invalid value encountered in scalar subtract
  

In [34]:
for x in sfr:
    sfr_median = np.median(x)
    print(sfr_median)

0.8349123147141413
0.013667906045965229
11.46736762917388
0.609096514679696
1.9494114997145515
0.3540260899372158
0.0
0.2712179951833926
0.00017142763595636955
0.0
0.0
0.0
3.5550421811015864
1.6244605579365186
0.03437438297826252
2.72766779732959
2.441033839367728
0.0
1.646748310446361
0.6231566823761335
0.4390285215630306
0.0
2.0673248452267825
0.7938819608664094
0.007121139692268359
0.3546546500752987
0.0
3.5228302151742574
0.515337286116418
2.324015648029625
0.0
0.793205054037363
0.0
0.0
1.5055427239870764
6.184515409932688
1.7556174404281533
0.7574748370097204
0.7478609496920089
1.3382425050573563
65.4516870441015
7.19256362815786
1.5219130674763193
3.698102571529161
2.340990298275398
18.42951979721606
0.42357671442721834
1.8441140413014858
1.086174797810047
0.0
0.0
1.892692518720651
2.8069407067075023
0.19960452228263956
0.2918888974269619
0.32879127578542894
0.1753666013868999
0.0
0.8765495547966283
2.2585445069365653
2.6930091055256824
0.1647212721099341
5.574526735256412
2.5906

- plot errorbars to data points of data, not just fill between of model
- print ratio, (16,50,84) to see if errorbar coversr above 1--> in that case is non-dust 
- have flag that checks ratio + upper 84th error
- redcued chi square to check if fits are decent (make sure to visually inspect each soruce in addition to reduced chi 
- plot mass
- sfr calculation (use halpha OR hbeta) - two diff sfr eqs

In [20]:
# RUBIES_and_CEERS_table['flux_intrinsic'] = flux_instrinsic_master
# RUBIES_and_CEERS_table['flux_error_corrected']= flux_error_master
# RUBIES_and_CEERS_table['dust_corrected']= flux_boolean
# RUBIES_and_CEERS_table['ratios']= ratios
# RUBIES_and_CEERS_table['which_ratio']= which_ratios_label
# RUBIES_and_CEERS_table['ratio_84th']= ratio_84_master

# #RUBIES_table['R VALUES']= R_master
# RUBIES_and_CEERS_table.write('MASTER_RUBIES_AND_CEERS_TABLE.fits',overwrite=True)